# 05 — Semitone Transposition

**Hypothesis (stress test):** The semitone shift maps white keys to black keys and vice versa non-uniformly. Pure rotors cannot implement it. This notebook finds where this breaks and surveys the norm² change across all 12 major keys.

In [1]:
from kingdon import Algebra
import numpy as np

alg = Algebra(7, 5, 1)
def mv(k): return alg.multivector({k: 1})

PITCH = {
    'C':  mv('e1'), 'D':  mv('e2'), 'E':  mv('e3'), 'F':  mv('e4'),
    'G':  mv('e5'), 'A':  mv('e6'), 'B':  mv('e7'),
    'Cs': mv('e8'), 'Ds': mv('e9'), 'Fs': mv('eA'), 'Gs': mv('eB'), 'As': mv('eC'),
}
OCTAVE = mv('e0')

SEMITONE_ORDER = ['C','Cs','D','Ds','E','F','Fs','G','Gs','A','As','B']
BLACK_KEYS = {'Cs', 'Ds', 'Fs', 'Gs', 'As'}

def norm2(x): return (x * ~x).e
def is_zero(x, tol=1e-10): return all(abs(float(v)) <= tol for v in x.values())
def blade_coeffs(x, tol=1e-10): return {alg.bin2canon[k]: float(v) for k, v in x.items() if abs(float(v)) > tol}

print('Setup ready.')

Setup ready.


In [2]:
# Show the white/black alternation in the chromatic scale
print('Signature of each pitch in chromatic order:')
for name in SEMITONE_ORDER:
    sig = '−1 (black)' if name in BLACK_KEYS else '+1 (white)'
    arrow = '→ ' + SEMITONE_ORDER[(SEMITONE_ORDER.index(name)+1)%12]
    target_sig = '−1' if SEMITONE_ORDER[(SEMITONE_ORDER.index(name)+1)%12] in BLACK_KEYS else '+1'
    crosses = '  ← crosses subspace' if (name in BLACK_KEYS) != (SEMITONE_ORDER[(SEMITONE_ORDER.index(name)+1)%12] in BLACK_KEYS) else ''
    print(f'  {name:3s} ({sig}) {arrow} ({target_sig}){crosses}')

Signature of each pitch in chromatic order:
  C   (+1 (white)) → Cs (−1)  ← crosses subspace
  Cs  (−1 (black)) → D (+1)  ← crosses subspace
  D   (+1 (white)) → Ds (−1)  ← crosses subspace
  Ds  (−1 (black)) → E (+1)  ← crosses subspace
  E   (+1 (white)) → F (+1)
  F   (+1 (white)) → Fs (−1)  ← crosses subspace
  Fs  (−1 (black)) → G (+1)  ← crosses subspace
  G   (+1 (white)) → Gs (−1)  ← crosses subspace
  Gs  (−1 (black)) → A (+1)  ← crosses subspace
  A   (+1 (white)) → As (−1)  ← crosses subspace
  As  (−1 (black)) → B (+1)  ← crosses subspace
  B   (+1 (white)) → C (+1)


In [3]:
# Every step (except E→F and B→C) crosses the positive/negative boundary
# Claim: pure rotors (which map positive→positive, negative→negative) cannot implement semitone shift
semitone_map = dict(zip(SEMITONE_ORDER, SEMITONE_ORDER[1:] + [SEMITONE_ORDER[0]]))

crossing_steps = [(a, semitone_map[a]) for a in SEMITONE_ORDER
                  if (a in BLACK_KEYS) != (semitone_map[a] in BLACK_KEYS)]
same_subspace  = [(a, semitone_map[a]) for a in SEMITONE_ORDER
                  if (a in BLACK_KEYS) == (semitone_map[a] in BLACK_KEYS)]

print(f'{len(crossing_steps)}/12 steps cross the positive/negative boundary:')
for a, b in crossing_steps:
    print(f'  {a} → {b}')
print(f'{len(same_subspace)}/12 steps stay within same subspace: {same_subspace}')
print()
print('✓ Pure rotors cannot implement semitone transposition (10 of 12 steps cross the boundary).')

10/12 steps cross the positive/negative boundary:
  C → Cs
  Cs → D
  D → Ds
  Ds → E
  F → Fs
  Fs → G
  G → Gs
  Gs → A
  A → As
  As → B
2/12 steps stay within same subspace: [('E', 'F'), ('B', 'C')]

✓ Pure rotors cannot implement semitone transposition (10 of 12 steps cross the boundary).


In [4]:
# Classify each step's bivector type
scalar1 = alg.multivector({'e': 1})

print(f'{"Step":<8}  {"B²":>4}  type')
print('-' * 30)
for a in SEMITONE_ORDER:
    b = semitone_map[a]
    B = PITCH[a] * PITCH[b]
    sq = (B * B).e
    kind = 'hyperbolic' if sq > 0 else 'circular' if sq < 0 else 'null'
    print(f'{a}→{b:<5}  {sq:>4}  {kind}')

Step        B²  type
------------------------------
C→Cs        1  hyperbolic
Cs→D         1  hyperbolic
D→Ds        1  hyperbolic
Ds→E         1  hyperbolic
E→F        -1  circular
F→Fs        1  hyperbolic
Fs→G         1  hyperbolic
G→Gs        1  hyperbolic
Gs→A         1  hyperbolic
A→As        1  hyperbolic
As→B         1  hyperbolic
B→C        -1  circular


In [5]:
# Hyperbolic rotor in the C→Cs plane: cannot achieve pure image
# Demonstrate: sandwich maps e1 → cosh(2t)·e1 + sinh(2t)·(±e8), never purely e8
scalar1 = alg.multivector({'e': 1})
B_C_Cs = PITCH['C'] * PITCH['Cs']

print(f'  {"t":>5}  {"e1(C) coeff":>14}  {"e8(Cs) coeff":>14}')
print('  ' + '-'*36)
for t in [0.0, 0.5, 1.0, 2.0, 4.0]:
    R = np.cosh(t) * scalar1 + np.sinh(t) * B_C_Cs
    img = R * PITCH['C'] * ~R
    bc = blade_coeffs(img, tol=0)
    print(f'  {t:>5.2f}  {bc.get("e1",0):>14.6f}  {bc.get("e8",0):>14.6f}')
print('  e1 → 0 only at t → ∞')

      t     e1(C) coeff    e8(Cs) coeff
  ------------------------------------
   0.00        1.000000        0.000000
   0.50        1.543081       -1.175201
   1.00        3.762196       -3.626860
   2.00       27.308233      -27.289917
   4.00     1490.479161    -1490.478826
  e1 → 0 only at t → ∞


In [6]:
# Norm² survey across all 12 major triads
print(f'{"Key":>4}  {"Triad":>12}  {"# black":>8}  {"‖·‖²":>6}  note')
print('-' * 48)
for i, root in enumerate(SEMITONE_ORDER):
    third = SEMITONE_ORDER[(i + 4) % 12]
    fifth = SEMITONE_ORDER[(i + 7) % 12]
    triad = PITCH[root] ^ PITCH[third] ^ PITCH[fifth]
    nblack = sum(1 for t in [root, third, fifth] if t in BLACK_KEYS)
    n2 = norm2(triad)
    note = '← reference' if root == 'C' else ''
    print(f'{root:>4}  {root+" "+third+" "+fifth:>12}  {nblack:>8}  {n2:>6}  {note}')

 Key         Triad   # black    ‖·‖²  note
------------------------------------------------
   C         C E G         0       1  ← reference
  Cs       Cs F Gs         2       1  
   D        D Fs A         1      -1  
  Ds       Ds G As         2       1  
   E        E Gs B         1      -1  
   F         F A C         0       1  
  Fs      Fs As Cs         3      -1  
   G         G B D         0       1  
  Gs       Gs C Ds         2       1  
   A        A Cs E         1      -1  
  As        As D F         1      -1  
   B       B Ds Fs         2       1  


## Discussion

**Confirmed failure (expected).**

10 of the 12 semitone steps cross the positive/negative subspace boundary. Each such step requires a hyperbolic bivector, and a real hyperbolic rotor can only asymptotically approach the other subspace — it never reaches it. The two same-subspace steps (E→F and B→C, both white→white) would be circular rotors, but they are embedded in a chain that requires hyperbolic steps.

The norm² survey shows that norm² = +1 only for the 3 all-white-key major triads (C, F, G) and norm² = −1 for all others. Semitone transposition cycles through these values, changing the metric at every step.

**This is the expected stress-test finding.** The algebra is deliberately not translation-invariant under chromatic transposition. The metric change is the algebraic encoding of key change: C-major is privileged, and transposing to any other key genuinely alters the algebraic structure.